# Milestone 15 — Streamlit Customer Intelligence & Retention Platform 📡

**Project:** Telecom Customer Intelligence  
**Notebook:** 15 of 16  
**Status:** ✅ Production-Ready

---

## Objective

Build an interactive, multi-page **Streamlit web application** that operationalises all
upstream analytics:

| Page | What it does |
|---|---|
| 🔍 **Customer Lookup** | Search any of the 10,000 customers by ID — profile, churn gauge, usage charts, retention recommendation |
| 🎯 **Retention Priority Explorer** | Filter/sort the full customer base by priority tier (Critical / High / Medium / Low), download results |
| 📊 **Model & Portfolio Summary** | Churn model performance, feature importance, campaign economics by tier |

**Inputs used:**
- `data/processed/customer_features_clustered.csv` (features + cluster labels)  
- `data/processed/customer_retention_priority.csv` (scores, tiers, recommendations)  
- `models/churn_feature_importance_rf.csv` / `churn_feature_importance_lr.csv`  
- `models/churn_model_comparison.csv`

> **Note on format:** Streamlit apps must run as Python scripts (`.py`) — they cannot execute  
> inside a Jupyter kernel. This notebook serves as the **milestone documentation and inline  
> visualisation demo** for the same analyses. The live app is at `app/app.py` and runs via  
> `streamlit run app/app.py`.

## 0 · Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
MODELS_DIR   = PROJECT_ROOT / "models"

# ── colour palette ──────────────────────────────────────────────────────────
TIER_COLORS = {
    "Critical": "#d62728",
    "High":     "#ff7f0e",
    "Medium":   "#2196F3",
    "Low":      "#4caf50",
}
TIER_ICONS  = {"Critical": "🔴", "High": "🟠", "Medium": "🔵", "Low": "🟢"}

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
# ── Load & merge data ────────────────────────────────────────────────────────
df_feat = pd.read_csv(DATA_DIR / "customer_features_clustered.csv")
df_pri  = pd.read_csv(DATA_DIR / "customer_retention_priority.csv")

df = df_feat.merge(
    df_pri[[
        "customer_id", "monthly_charges_clean", "estimated_historical_revenue",
        "annualized_revenue", "value_score", "predicted_churn_risk",
        "expected_revenue_at_risk", "retention_priority", "priority_tier",
        "retention_recommendation", "retention_rank"
    ]],
    on="customer_id", how="left"
)
df["monthly_charges"] = df["monthly_charges_clean"].fillna(df["monthly_charges"])
df.drop(columns=["monthly_charges_clean"], inplace=True)

# ── Load model artefacts ─────────────────────────────────────────────────────
fi_rf     = pd.read_csv(MODELS_DIR / "churn_feature_importance_rf.csv")
fi_lr     = pd.read_csv(MODELS_DIR / "churn_feature_importance_lr.csv")
model_cmp = pd.read_csv(MODELS_DIR / "churn_model_comparison.csv")

for fi in (fi_rf, fi_lr):
    col = "feature"
    fi["feature_clean"] = (fi[col]
        .str.replace("num__", "", regex=False)
        .str.replace("cat__", "", regex=False)
        .str.replace("_", " ", regex=False)
        .str.title())

print(f"Customers loaded  : {len(df):,}")
print(f"Priority tiers    : {df['priority_tier'].value_counts().to_dict()}")
print(f"Null priority_tier: {df['priority_tier'].isna().sum()}")
df.head(3)

Customers loaded  : 10,000
Priority tiers    : {'Low': 4000, 'Medium': 3000, 'High': 2000, 'Critical': 1000}
Null priority_tier: 0


,customer_id,age,gender,region,contract_type,monthly_charges,age_missing,tenure_months,signed_up_after_window_start,call_minutes_mean,call_minutes_std,call_minutes_min,call_minutes_max,call_minutes_slope,call_minutes_recent_vs_early,call_minutes_cv,data_usage_gb_mean,data_usage_gb_std,data_usage_gb_min,data_usage_gb_max,...,sms_count_recent_vs_early,sms_count_cv,complaints_total,complaints_mean,complaint_months,complaints_recent3,active_months,data_gb_per_charge,minutes_per_charge,churn,cluster,estimated_historical_revenue,annualized_revenue,value_score,predicted_churn_risk,expected_revenue_at_risk,retention_priority,priority_tier,retention_recommendation,retention_rank
0,CUST_0,NaN,Male,Urban,Yearly,48.1800,1,33,0,493.4122,53.7263,434.8200,590.2600,3.3090,1.0639,0.1089,9.2856,2.8133,5.9900,14.1600,...,1.4828,0.4599,6,0.6667,5,0,9,0.1927,10.2410,0,1,1589.9400,578.1600,43.0950,0.0895,51.7652,56.0975,Medium,Send a personalized engagement message,4123
1,CUST_1,NaN,Female,Suburban,Prepaid,38.9400,1,47,0,457.1944,52.6004,396.8800,564.0700,-2.3478,1.0056,0.1151,10.0833,3.6994,3.4500,15.6800,...,0.8830,0.4433,8,0.8889,6,2,9,0.2589,11.7410,0,1,1830.1800,467.2800,42.2900,0.0644,30.1127,35.2400,Low,Continue regular customer communication,7363
2,CUST_10,NaN,Female,Urban,Prepaid,47.8100,1,20,0,617.4189,38.4198,534.4300,658.8400,-6.1312,0.9234,0.0622,8.2233,5.0537,0.0000,12.5300,...,0.5455,0.6465,11,1.2222,7,5,9,0.1720,12.9140,0,1,956.2000,573.7200,32.9325,0.0887,50.8889,50.2413,Medium,Send a personalized engagement message,5097


---
## Page 1 · 🔍 Customer Lookup — Inline Demo

The live app lets users search any Customer ID and renders:
- Header banner with tier badge and retention rank  
- 5 KPI cards (monthly charge, tenure, churn risk, value score, revenue at risk)  
- Profile + financial tables  
- Churn risk gauge (Plotly Indicator)  
- Usage behaviour table  
- Contextual retention recommendation  
- 3 distribution histograms with customer position highlighted  

Here we demonstrate the visualisations for a sample **Critical-tier** customer.

In [3]:
# ── Pick a sample Critical customer ─────────────────────────────────────────
sample_id = (df[df["priority_tier"] == "Critical"]
             .sort_values("predicted_churn_risk", ascending=False)
             .iloc[0]["customer_id"])

row = df[df["customer_id"] == sample_id].iloc[0]
tier       = row["priority_tier"]
tier_color = TIER_COLORS[tier]
churn_risk = float(row["predicted_churn_risk"])
risk_color = "#d62728" if churn_risk > 0.15 else "#ff7f0e" if churn_risk > 0.10 else "#4caf50"

print(f"Demo customer : {sample_id}")
print(f"Tier          : {TIER_ICONS[tier]} {tier}")
print(f"Retention rank: #{int(row['retention_rank']):,}")
print(f"Churn risk    : {churn_risk*100:.2f}%")
print(f"Value score   : {row['value_score']:.1f}/100")
print(f"Monthly charge: ${row['monthly_charges']:.2f}")
print(f"Recommendation: {row['retention_recommendation']}")

Demo customer : CUST_5007
Tier          : 🔴 Critical
Retention rank: #91
Churn risk    : 65.62%
Value score   : 81.8/100
Monthly charge: $65.25
Recommendation: Personal call and resolve any service issues


In [4]:
# ── KPI Summary Table ────────────────────────────────────────────────────────
kpis = pd.DataFrame({
    "KPI": [
        "Monthly Charges", "Tenure", "Churn Risk Score",
        "Value Score", "Expected Revenue at Risk"
    ],
    "Value": [
        f"${row['monthly_charges']:.2f}",
        f"{int(row['tenure_months'])} months (~{row['tenure_months']/12:.1f} yrs)",
        f"{churn_risk*100:.1f}%  ({'⚠️ HIGH' if churn_risk > 0.15 else '🟡 MEDIUM' if churn_risk > 0.10 else '✅ LOW'})",
        f"{row['value_score']:.1f} / 100",
        f"${row['expected_revenue_at_risk']:.2f}"
    ]
})
print(kpis.to_string(index=False))

                     KPI                Value
         Monthly Charges               $65.25
                  Tenure 59 months (~4.9 yrs)
        Churn Risk Score     65.6%  (⚠️ HIGH)
             Value Score           81.8 / 100
Expected Revenue at Risk              $513.77


In [5]:
# ── Churn Risk Gauge ─────────────────────────────────────────────────────────
fig_gauge = go.Figure(go.Indicator(
    mode="gauge+number",
    value=churn_risk * 100,
    number={"suffix": "%", "font": {"size": 36, "color": "#e0e0e0"}},
    gauge={
        "axis": {"range": [0, 100], "tickfont": {"color": "#8b949e"}},
        "bar":  {"color": risk_color, "thickness": 0.3},
        "bgcolor": "#1e2030",
        "borderwidth": 0,
        "steps": [
            {"range": [0, 8],    "color": "#1e3a1e"},
            {"range": [8, 15],   "color": "#3a2e10"},
            {"range": [15, 100], "color": "#3a1010"},
        ],
        "threshold": {
            "line": {"color": "white", "width": 2},
            "thickness": 0.75,
            "value": 7.86,
        }
    },
    title={
        "text": (f"Customer {sample_id} — OOF Churn Risk<br>"
                 "<sup>White line = population baseline (7.86%)</sup>"),
        "font": {"color": "#8b949e", "size": 13}
    },
))
fig_gauge.update_layout(
    height=320,
    paper_bgcolor="#0f1117", plot_bgcolor="#0f1117",
    font=dict(color="#c9d1d9"),
    margin=dict(t=80, b=20, l=60, r=60),
)
fig_gauge.show()

In [6]:
# ── Customer Profile ─────────────────────────────────────────────────────────
cluster_map   = {0: "Segment A — High Usage / Low Complaints",
                 1: "Segment B — Moderate Usage"}
cluster_label = cluster_map.get(int(row.get("cluster", 0)), "Unknown")
age_str       = f"{int(row['age'])} yrs" if pd.notna(row.get("age")) else "Unknown"

profile = pd.DataFrame({
    "Field": ["Customer ID", "Gender", "Age", "Region", "Contract",
              "Cluster / Segment", "Tenure", "Active Months",
              "Complaints (Total)", "Recent Complaints (3M)"],
    "Value": [
        sample_id,
        str(row.get("gender", "N/A")),
        age_str,
        str(row.get("region", "N/A")),
        str(row.get("contract_type", "N/A")),
        cluster_label,
        f"{int(row['tenure_months'])} months",
        str(int(row.get("active_months", 0))),
        str(int(row.get("complaints_total", 0))),
        str(int(row.get("complaints_recent3", 0))),
    ]
})
print(profile.to_string(index=False))

                 Field                      Value
           Customer ID                  CUST_5007
                Gender                     Female
                   Age                     49 yrs
                Region                      Rural
              Contract                    Prepaid
     Cluster / Segment Segment B — Moderate Usage
                Tenure                  59 months
         Active Months                          9
    Complaints (Total)                          5
Recent Complaints (3M)                          1


In [7]:
# ── Usage Distribution Histograms ────────────────────────────────────────────
fig_hist = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Avg Monthly Call Minutes", "Avg Monthly Data (GB)", "Avg Monthly SMS"],
)
configs = [
    ("call_minutes_mean", "min", 1),
    ("data_usage_gb_mean", "GB",  2),
    ("sms_count_mean",    "msgs", 3),
]
for col_name, unit, col_idx in configs:
    vals   = df[col_name].dropna()
    cust_v = row.get(col_name, 0)

    fig_hist.add_trace(
        go.Histogram(x=vals, nbinsx=40, marker_color="#2a3f5f",
                     opacity=0.8, showlegend=False),
        row=1, col=col_idx
    )
    fig_hist.add_vline(
        x=cust_v, line_color="#ff7f0e", line_width=2.5,
        annotation_text=f"{cust_v:.1f} {unit}",
        annotation_font_color="#ff7f0e",
        row=1, col=col_idx
    )

fig_hist.update_layout(
    height=300,
    paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
    font=dict(color="#c9d1d9", size=11),
    title=dict(
        text=f"Customer {sample_id} Position in Population Distribution (orange line)",
        font=dict(color="#8b949e", size=12)
    ),
    margin=dict(t=60, b=30, l=40, r=20),
)
fig_hist.update_xaxes(gridcolor="#2a2d3e")
fig_hist.update_yaxes(gridcolor="#2a2d3e")
fig_hist.show()

---
## Page 2 · 🎯 Retention Priority Explorer — Inline Demo

The live app provides interactive multi-select filters for Priority Tier,
Contract Type, Region, Churn Status, and numeric range sliders.

Below we demonstrate the **tier summary statistics** and **cohort analysis charts**.

In [8]:
# ── Tier Summary Statistics ──────────────────────────────────────────────────
tier_stats = (df.groupby("priority_tier").agg(
    Customers        = ("customer_id",          "count"),
    Avg_Churn_Risk   = ("predicted_churn_risk",  "mean"),
    Avg_Value_Score  = ("value_score",            "mean"),
    Total_Rev_at_Risk= ("expected_revenue_at_risk","sum"),
    Actual_Churn_Rate= ("churn",                  "mean"),
    Avg_Tenure_Mo    = ("tenure_months",           "mean"),
    Avg_Monthly_Chg  = ("monthly_charges",         "mean"),
).reindex(["Critical", "High", "Medium", "Low"]))

tier_stats["Avg_Churn_Risk_pct"] = (tier_stats["Avg_Churn_Risk"]   * 100).round(2)
tier_stats["Actual_Churn_pct"]   = (tier_stats["Actual_Churn_Rate"] * 100).round(2)
tier_stats["Total_Rev_at_Risk"]  = tier_stats["Total_Rev_at_Risk"].round(2)

print(tier_stats[[
    "Customers", "Avg_Churn_Risk_pct", "Avg_Value_Score",
    "Total_Rev_at_Risk", "Actual_Churn_pct", "Avg_Tenure_Mo", "Avg_Monthly_Chg"
]].to_string())

               Customers  Avg_Churn_Risk_pct  Avg_Value_Score  Total_Rev_at_Risk  Actual_Churn_pct  Avg_Tenure_Mo  Avg_Monthly_Chg
priority_tier                                                                                                                     
Critical            1000             11.9400          77.7458         92794.4200           21.2000        56.0730          66.4337
High                2000              9.8900          61.9130        130953.9200           13.6500        43.5260          58.5547
Medium              3000              7.6500          52.5015        135449.3700            6.4700        38.5983          52.9126
Low                 4000              5.8800          35.2434        104106.1400            2.6800        33.3162          39.4303


In [9]:
# ── Filter Demo: Critical + High tiers ──────────────────────────────────────
selected_tiers = ["Critical", "High"]
filtered = df[df["priority_tier"].isin(selected_tiers)].copy()

print(f"Filtered cohort : {len(filtered):,} customers")
print(f"Tiers           : {filtered['priority_tier'].value_counts().to_dict()}")
print()

display_cols = {
    "retention_rank":          "Rank",
    "customer_id":             "Customer ID",
    "priority_tier":           "Tier",
    "monthly_charges":         "Monthly ($)",
    "tenure_months":           "Tenure (Mo)",
    "contract_type":           "Contract",
    "region":                  "Region",
    "value_score":             "Value Score",
    "predicted_churn_risk":    "Churn Risk",
    "expected_revenue_at_risk":"Rev. at Risk ($)",
    "churn":                   "Actual Churn",
}
table_df = filtered[[c for c in display_cols if c in filtered.columns]].rename(columns=display_cols)
table_df["Churn Risk"]      = (table_df["Churn Risk"] * 100).round(1).astype(str) + "%"
table_df["Value Score"]     = table_df["Value Score"].round(1)
table_df["Monthly ($)"]     = table_df["Monthly ($)"].round(2)
table_df["Rev. at Risk ($)"]= table_df["Rev. at Risk ($)"].round(2)
table_df.sort_values("Rank").head(10)

Filtered cohort : 3,000 customers
Tiers           : {'High': 2000, 'Critical': 1000}



,Rank,Customer ID,Tier,Monthly ($),Tenure (Mo),Contract,Region,Value Score,Churn Risk,Rev. at Risk ($),Actual Churn
3973,1,CUST_4574,Critical,92.3600,72,Yearly,Rural,99.5000,10.2%,113.3400,0
3130,2,CUST_3815,Critical,85.8000,69,Yearly,Suburban,96.6000,54.5%,561.2300,1
5448,3,CUST_5901,Critical,93.4000,68,Monthly,Urban,96.2000,27.0%,302.5400,1
8603,4,CUST_8741,Critical,94.1200,68,Prepaid,Rural,96.2000,13.4%,151.0000,0
1282,5,CUST_2151,Critical,82.2700,71,Monthly,Urban,97.8000,11.0%,108.2300,0
7669,6,CUST_7900,Critical,74.6100,69,Prepaid,Suburban,94.4000,36.7%,329.0200,1
2214,7,CUST_2991,Critical,76.2600,69,Yearly,Urban,95.0000,20.6%,188.8600,0
368,8,CUST_1329,Critical,88.1900,68,Monthly,Suburban,96.0000,11.4%,120.3200,0
215,9,CUST_1191,Critical,87.5100,69,Monthly,Urban,96.7000,10.1%,105.8300,0
8950,10,CUST_9053,Critical,70.6200,70,Prepaid,Rural,93.6000,29.3%,248.1100,1


In [10]:
# ── Cohort Analysis Charts ───────────────────────────────────────────────────
fig_cohort = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Tier Distribution",
        "Churn Risk Distribution by Tier",
        "Value Score vs. Churn Risk",
    ],
    specs=[[{"type":"pie"}, {"type":"xy"}, {"type":"xy"}]],
)

# Pie
tier_count = (filtered["priority_tier"]
              .value_counts()
              .reindex(["Critical", "High", "Medium", "Low"])
              .fillna(0))
fig_cohort.add_trace(
    go.Pie(
        values=tier_count.values,
        labels=tier_count.index,
        marker_colors=[TIER_COLORS[t] for t in tier_count.index],
        hole=0.45,
        textinfo="label+percent",
        showlegend=False,
    ),
    row=1, col=1
)

# Box
for tier_name in ["Critical", "High", "Medium", "Low"]:
    sub = filtered[filtered["priority_tier"] == tier_name]["predicted_churn_risk"]
    if len(sub):
        fig_cohort.add_trace(
            go.Box(y=sub, name=tier_name, marker_color=TIER_COLORS[tier_name],
                   showlegend=False),
            row=1, col=2
        )

# Scatter (sample up to 500 for readability)
samp = filtered.sample(min(500, len(filtered)), random_state=42)
for tier_name in ["Critical", "High", "Medium", "Low"]:
    sub = samp[samp["priority_tier"] == tier_name]
    if len(sub):
        fig_cohort.add_trace(
            go.Scatter(
                x=sub["value_score"], y=sub["predicted_churn_risk"],
                mode="markers",
                marker=dict(color=TIER_COLORS[tier_name], size=5, opacity=0.6),
                name=tier_name,
                showlegend=(True if tier_name in ["Critical","High"] else False),
            ),
            row=1, col=3
        )

fig_cohort.update_layout(
    height=380, paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
    font=dict(color="#c9d1d9"),
    title=dict(text=f"Cohort Analysis — {', '.join(selected_tiers)} Priority Customers",
               font=dict(color="#8b949e")),
    margin=dict(t=80, b=30),
)
fig_cohort.update_xaxes(gridcolor="#2a2d3e")
fig_cohort.update_yaxes(gridcolor="#2a2d3e")
fig_cohort.update_yaxes(tickformat=".0%", col=2)
fig_cohort.update_yaxes(tickformat=".0%", col=3)
fig_cohort.show()

---
## Page 3 · 📊 Model & Portfolio Summary — Inline Demo

This section reproduces all charts from the app's third page:

1. Portfolio financial overview KPIs  
2. Churn model comparison table (OOF-tuned thresholds)  
3. Random Forest feature importance bar chart  
4. Retention campaign economics by tier

In [11]:
# ── Portfolio Financial KPIs ─────────────────────────────────────────────────
total_annual   = df["annualized_revenue"].sum()
total_exp_loss = df["expected_revenue_at_risk"].sum()
total_hist     = df["estimated_historical_revenue"].sum()
churn_base     = df["churn"].mean()
imputed_n      = 54

portfolio_kpis = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Annualized Portfolio Billing",
        "Expected Revenue Loss at Risk",
        "Est. Historical Revenue (Cumulative)",
        "Baseline Churn Rate",
        "Avg Monthly Charge (per customer)",
    ],
    "Value": [
        f"{len(df):,}  ({imputed_n} imputed charges)",
        f"${total_annual/1e6:.2f}M  (portfolio run-rate)",
        f"${total_exp_loss/1e3:.1f}K  ({total_exp_loss/total_annual*100:.2f}% of billing)",
        f"${total_hist/1e6:.1f}M",
        f"{churn_base*100:.2f}%  (786 churned / 10,000)",
        f"${df['monthly_charges'].mean():.2f}",
    ]
})
print(portfolio_kpis.to_string(index=False))

                              Metric                         Value
                     Total Customers  10,000  (54 imputed charges)
        Annualized Portfolio Billing  $6.00M  (portfolio run-rate)
       Expected Revenue Loss at Risk   $463.3K  (7.72% of billing)
Est. Historical Revenue (Cumulative)                        $19.6M
                 Baseline Churn Rate 7.86%  (786 churned / 10,000)
   Avg Monthly Charge (per customer)                        $50.00


In [12]:
# ── Churn Model Comparison Table ─────────────────────────────────────────────
tuned = model_cmp[model_cmp["Decision Rule"] == "Train-OOF Tuned"].copy()
cols_show = ["Model", "ROC-AUC", "PR-AUC (AP)", "Precision", "Recall", "F1", "Brier", "Threshold"]
available = [c for c in cols_show if c in tuned.columns]
print(tuned[available].round(4).to_string(index=False))

              Model  ROC-AUC  PR-AUC (AP)  Precision  Recall     F1  Brier  Threshold
Logistic Regression   0.4826       0.0775     0.0785  1.0000 0.1456 0.0724     0.0741
      Random Forest   0.4851       0.0769     0.0786  1.0000 0.1458 0.0724     0.0712
            XGBoost   0.4921       0.0765     0.0779  0.9682 0.1443 0.0724     0.0736


In [13]:
# ── Feature Importance — Random Forest (Top 20) ──────────────────────────────
top_n    = 20
fi_plot  = fi_rf.head(top_n).sort_values("importance")

fig_fi = px.bar(
    fi_plot, x="importance", y="feature_clean",
    orientation="h", color="importance",
    color_continuous_scale="Blues",
    title=f"Top {top_n} Churn Predictors — Random Forest Feature Importance",
    labels={"importance": "Importance Score", "feature_clean": "Feature"},
)
fig_fi.update_layout(
    paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
    font=dict(color="#c9d1d9"),
    coloraxis_showscale=False,
    height=max(380, top_n * 24),
    margin=dict(t=60, b=20, l=10, r=30),
    xaxis=dict(gridcolor="#2a2d3e"),
    yaxis=dict(gridcolor="#2a2d3e"),
)
fig_fi.show()

In [14]:
# ── Top 10 Features Side-by-Side ─────────────────────────────────────────────
print("Random Forest — Top 10 by Importance:")
print(fi_rf[["feature_clean","importance"]].head(10).rename(
    columns={"feature_clean":"Feature","importance":"Importance"}).round(4).to_string(index=False))
print()
print("Logistic Regression — Top 10 by |Coefficient|:")
print(fi_lr[["feature_clean","coefficient"]].head(10).rename(
    columns={"feature_clean":"Feature","coefficient":"Coefficient"}).round(4).to_string(index=False))

Random Forest — Top 10 by Importance:
                      Feature  Importance
             Call Minutes Std      0.0455
              Call Minutes Cv      0.0450
           Minutes Per Charge      0.0443
           Call Minutes Slope      0.0411
 Call Minutes Recent Vs Early      0.0407
               Sms Count Mean      0.0406
            Data Usage Gb Min      0.0401
              Sms Count Slope      0.0400
           Data Usage Gb Mean      0.0395
Data Usage Gb Recent Vs Early      0.0392

Logistic Regression — Top 10 by |Coefficient|:
             Feature  Coefficient
  Minutes Per Charge      -0.3555
  Data Gb Per Charge       0.3145
   Call Minutes Mean       0.2184
    Call Minutes Std       0.1912
    Data Usage Gb Cv      -0.1713
   Data Usage Gb Std       0.1651
        Region Urban       0.1608
     Call Minutes Cv      -0.1604
    Call Minutes Max      -0.1515
Contract Type Yearly       0.1409


In [15]:
# ── Retention Campaign Economics ─────────────────────────────────────────────
outreach_costs = {"Critical": 25.0, "High": 12.0, "Medium": 3.0, "Low": 0.5}

econ = df.groupby("priority_tier").agg(
    customers          = ("customer_id",             "count"),
    total_billing      = ("annualized_revenue",       "sum"),
    exp_revenue_at_risk= ("expected_revenue_at_risk", "sum"),
    avg_value_score    = ("value_score",              "mean"),
    actual_churn       = ("churn",                    "sum"),
).reindex(["Critical", "High", "Medium", "Low"])

econ["outreach_cost_per_cust"] = econ.index.map(outreach_costs)
econ["total_outreach_cost"]    = econ["customers"] * econ["outreach_cost_per_cust"]
econ["expected_saved_margin"]  = econ["exp_revenue_at_risk"] * 0.60 * 0.15
econ["net_benefit"]            = econ["expected_saved_margin"] - econ["total_outreach_cost"]

econ_display = econ[[
    "customers", "total_billing", "exp_revenue_at_risk",
    "total_outreach_cost", "expected_saved_margin", "net_benefit"
]].rename(columns={
    "customers":            "Customers",
    "total_billing":        "Annual Billing ($)",
    "exp_revenue_at_risk":  "Rev. at Risk ($)",
    "total_outreach_cost":  "Campaign Cost ($)",
    "expected_saved_margin":"Saved Margin ($)",
    "net_benefit":          "Net Benefit ($)",
}).round(2)

print("Campaign Economics by Priority Tier")
print("Assumptions: 60% gross margin, 15% offer acceptance rate")
print()
print(econ_display.to_string())

Campaign Economics by Priority Tier
Assumptions: 60% gross margin, 15% offer acceptance rate

               Customers  Annual Billing ($)  Rev. at Risk ($)  Campaign Cost ($)  Saved Margin ($)  Net Benefit ($)
priority_tier                                                                                                       
Critical            1000         797204.4000        92794.4200         25000.0000         8351.5000      -16648.5000
High                2000        1405312.2000       130953.9200         24000.0000        11785.8500      -12214.1500
Medium              3000        1904852.8800       135449.3700          9000.0000        12190.4400        3190.4400
Low                 4000        1892652.6000       104106.1400          2000.0000         9369.5500        7369.5500


In [16]:
# ── Net Benefit Bar Chart ────────────────────────────────────────────────────
fig_nb = px.bar(
    econ.reset_index(), x="priority_tier", y="net_benefit",
    color="priority_tier", color_discrete_map=TIER_COLORS,
    title="Expected Net Campaign Benefit by Priority Tier ($)",
    labels={"priority_tier": "Priority Tier", "net_benefit": "Net Benefit ($)"},
    category_orders={"priority_tier": ["Critical", "High", "Medium", "Low"]},
    text_auto=".0f",
)
fig_nb.update_layout(
    paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
    font=dict(color="#c9d1d9"),
    showlegend=False, height=380,
    margin=dict(t=60, b=30),
    xaxis=dict(gridcolor="#2a2d3e"),
    yaxis=dict(gridcolor="#2a2d3e"),
)
fig_nb.update_traces(textfont_color="#ffffff", textposition="outside")
fig_nb.show()

---
## App Architecture

```
app/
└── app.py              ← Main Streamlit entry-point (719 lines)
    ├── Page config & global CSS (dark theme, metric cards, tier badges)
    ├── @st.cache_data  load_data()          → merges features + retention CSVs
    ├── @st.cache_data  load_model_data()    → loads feature importance + comparison
    ├── Sidebar         → navigation radio, live data snapshot KPIs
    ├── Page 1 — Customer Lookup
    │   ├── Search bar + 8 quick-example buttons
    │   ├── Header banner (tier badge, retention rank)
    │   ├── 5-column KPI row
    │   ├── Profile table + Financial table
    │   ├── Churn risk gauge + Usage behaviour table
    │   ├── Retention recommendation box
    │   └── 3 usage distribution histograms with customer marker
    ├── Page 2 — Retention Priority Explorer
    │   ├── 4 tier summary cards (count, avg risk, rev at risk, churn rate)
    │   ├── Filters: Priority Tier (multiselect), Contract, Region, Churn Status
    │   ├── Sliders: Monthly Charges, Churn Risk %
    │   ├── Sortable results table (top 200) + CSV download button
    │   └── 3 cohort charts (donut, box, scatter)
    └── Page 3 — Model & Portfolio Summary
        ├── 5 portfolio KPI cards
        ├── Model comparison table (OOF-tuned)
        ├── Model diagnostic warning
        ├── RF feature importance bar chart (adjustable top-N slider)
        ├── RF + LR feature tables
        └── Campaign economics table + net benefit bar chart
```

### Tech Stack
| Component | Library | Version |
|---|---|---|
| Web framework | Streamlit | 1.65.0 |
| Interactive charts | Plotly | 7.1.0 |
| Data manipulation | Pandas / NumPy | — |
| Model artifacts | scikit-learn / joblib | — |

---
## Complete Streamlit Application Code (`app/app.py`)

The complete production source code for the interactive Streamlit application is embedded below.
Running this cell will write or update `../app/app.py` directly from this notebook.

In [17]:
%%writefile ../app/app.py
"""
Telecom Customer Intelligence & Retention Platform
Streamlit Application — Milestone 15
"""

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# ─── Page Config ────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Telecom Intelligence Platform",
    page_icon="📡",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─── Constants ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

TIER_COLORS = {
    "Critical": "#d62728",
    "High":     "#ff7f0e",
    "Medium":   "#2196F3",
    "Low":      "#4caf50",
}
TIER_ICONS = {
    "Critical": "🔴",
    "High":     "🟠",
    "Medium":   "🔵",
    "Low":      "🟢",
}

# ─── Custom CSS ─────────────────────────────────────────────────────────────────
st.markdown("""
<style>
/* ---------- global ---------- */
html, body, [data-testid="stAppViewContainer"] {
    font-family: 'Inter', 'Segoe UI', sans-serif;
    background: #0f1117;
    color: #e0e0e0;
}

/* ---------- sidebar ---------- */
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #1a1d2e 0%, #0f1117 100%);
    border-right: 1px solid #2a2d3e;
}
[data-testid="stSidebar"] .stRadio label,
[data-testid="stSidebar"] p,
[data-testid="stSidebar"] label {
    color: #c9d1d9 !important;
}

/* ---------- metric cards ---------- */
.metric-card {
    background: linear-gradient(135deg, #1e2030 0%, #16192a 100%);
    border-radius: 12px;
    padding: 18px 22px;
    border: 1px solid #2a2d3e;
    margin-bottom: 10px;
}
.metric-card .label  { font-size: 0.78rem; color: #8b949e; text-transform: uppercase; letter-spacing: 0.06em; }
.metric-card .value  { font-size: 1.8rem; font-weight: 700; margin-top: 4px; }
.metric-card .sub    { font-size: 0.82rem; color: #8b949e; margin-top: 4px; }

/* ---------- tier badges ---------- */
.badge-critical { background:#d62728;  color:#fff; padding:4px 14px; border-radius:20px; font-weight:700; font-size:0.85rem; }
.badge-high     { background:#ff7f0e;  color:#fff; padding:4px 14px; border-radius:20px; font-weight:700; font-size:0.85rem; }
.badge-medium   { background:#2196F3;  color:#fff; padding:4px 14px; border-radius:20px; font-weight:700; font-size:0.85rem; }
.badge-low      { background:#4caf50;  color:#fff; padding:4px 14px; border-radius:20px; font-weight:700; font-size:0.85rem; }

/* ---------- section headers ---------- */
.section-header {
    font-size: 1.1rem;
    font-weight: 600;
    color: #58a6ff;
    border-bottom: 1px solid #2a2d3e;
    padding-bottom: 6px;
    margin-bottom: 14px;
    margin-top: 10px;
}

/* ---------- recommendation box ---------- */
.rec-box {
    background: linear-gradient(135deg, #0d1117 0%, #161b22 100%);
    border-left: 4px solid #58a6ff;
    border-radius: 8px;
    padding: 14px 18px;
    font-size: 0.93rem;
    line-height: 1.6;
    color: #c9d1d9;
}

/* ---------- dataframe overrides ---------- */
[data-testid="stDataFrame"] { border-radius: 10px; overflow: hidden; }

/* ---------- progress bar ---------- */
.stProgress > div > div > div { border-radius: 4px; }
</style>
""", unsafe_allow_html=True)


# ─── Data Loaders ───────────────────────────────────────────────────────────────
@st.cache_data(show_spinner="Loading customer data…")
def load_data():
    df_feat = pd.read_csv(DATA_DIR / "customer_features_clustered.csv")
    df_pri  = pd.read_csv(DATA_DIR / "customer_retention_priority.csv")

    # Merge — keep clean versions from retention file
    df = df_feat.merge(
        df_pri[["customer_id", "monthly_charges_clean", "estimated_historical_revenue",
                "annualized_revenue", "value_score", "predicted_churn_risk",
                "expected_revenue_at_risk", "retention_priority", "priority_tier",
                "retention_recommendation", "retention_rank"]],
        on="customer_id", how="left"
    )
    # Use original monthly_charges where clean version is available
    df["monthly_charges"] = df["monthly_charges_clean"].fillna(df["monthly_charges"])
    df.drop(columns=["monthly_charges_clean"], inplace=True)
    return df


@st.cache_data(show_spinner="Loading model artifacts…")
def load_model_data():
    fi_rf    = pd.read_csv(MODELS_DIR / "churn_feature_importance_rf.csv")
    fi_lr    = pd.read_csv(MODELS_DIR / "churn_feature_importance_lr.csv")
    model_cmp = pd.read_csv(MODELS_DIR / "churn_model_comparison.csv")

    # Clean feature names
    fi_rf["feature_clean"] = (fi_rf["feature"]
        .str.replace("num__", "", regex=False)
        .str.replace("cat__", "", regex=False)
        .str.replace("_", " ", regex=False).str.title())

    fi_lr["feature_clean"] = (fi_lr["feature"]
        .str.replace("num__", "", regex=False)
        .str.replace("cat__", "", regex=False)
        .str.replace("_", " ", regex=False).str.title())

    return fi_rf, fi_lr, model_cmp


# ─── Sidebar Navigation ─────────────────────────────────────────────────────────
with st.sidebar:
    st.image("https://img.icons8.com/color/80/satellite-signal.png", width=60)
    st.markdown("## 📡 Telecom Intelligence")
    st.markdown("**Customer Intelligence & Retention Platform**")
    st.markdown("---")

    page = st.radio(
        "Navigate",
        options=["🔍 Customer Lookup", "🎯 Retention Priority Explorer", "📊 Model & Portfolio Summary"],
        index=0,
        label_visibility="collapsed"
    )

    st.markdown("---")
    st.markdown("**Data Snapshot**")

    df = load_data()

    total = len(df)
    churn_rate = df["churn"].mean() * 100
    critical_n = (df["priority_tier"] == "Critical").sum()
    high_n     = (df["priority_tier"] == "High").sum()

    st.metric("Total Customers", f"{total:,}")
    st.metric("Overall Churn Rate", f"{churn_rate:.2f}%")
    st.metric("🔴 Critical Priority", f"{critical_n:,}")
    st.metric("🟠 High Priority", f"{high_n:,}")

    st.markdown("---")
    st.caption("Milestone 15 · Streamlit App")


# ════════════════════════════════════════════════════════════════════════════════
# PAGE 1 — Customer Lookup
# ════════════════════════════════════════════════════════════════════════════════
if page == "🔍 Customer Lookup":
    st.title("🔍 Customer Lookup")
    st.markdown("Search any customer to view their full profile, churn risk assessment, and retention recommendation.")

    # Search bar
    col_search, col_btn = st.columns([4, 1])
    with col_search:
        query = st.text_input(
            "Customer ID",
            placeholder="e.g. CUST_1234",
            label_visibility="collapsed"
        )
    with col_btn:
        st.markdown("<br>", unsafe_allow_html=True)
        search = st.button("Search", use_container_width=True, type="primary")

    # Quick-pick random examples
    st.caption("Quick examples:")
    ex_cols = st.columns(8)
    examples = (df.groupby("priority_tier")
                .apply(lambda g: g.sample(2, random_state=42))
                .reset_index(drop=True)["customer_id"].tolist())
    example_clicked = None
    for i, ex in enumerate(examples[:8]):
        with ex_cols[i % 8]:
            if st.button(ex, key=f"ex_{i}", use_container_width=True):
                example_clicked = ex
                query = ex

    customer_id = example_clicked if example_clicked else query.strip()

    if not customer_id:
        st.info("Enter a Customer ID above or click one of the quick examples.")
        st.stop()

    row = df[df["customer_id"] == customer_id]
    if row.empty:
        st.error(f"No customer found with ID **{customer_id}**. Please check the ID and try again.")
        st.stop()

    row = row.iloc[0]
    tier = str(row.get("priority_tier", "Unknown"))
    tier_color = TIER_COLORS.get(tier, "#666")
    tier_icon  = TIER_ICONS.get(tier, "⚪")

    # ── Header Banner ──
    st.markdown(f"""
    <div style="background:linear-gradient(135deg,#1a1d2e,#0f1117);
                border-radius:14px;padding:20px 28px;
                border:1px solid {tier_color}44;margin-bottom:18px;">
        <span style="font-size:1.6rem;font-weight:800;color:#e0e0e0;">{customer_id}</span>
        &nbsp;&nbsp;
        <span class="badge-{tier.lower()}">{tier_icon} {tier} Priority</span>
        &nbsp;&nbsp;
        <span style="color:#8b949e;font-size:0.9rem;">Retention Rank #{int(row.get('retention_rank', 0)):,} of 10,000</span>
    </div>
    """, unsafe_allow_html=True)

    # ── Top KPI Row ──
    k1, k2, k3, k4, k5 = st.columns(5)

    def kpi(col, label, value, sub="", color="#58a6ff"):
        col.markdown(f"""
        <div class="metric-card">
            <div class="label">{label}</div>
            <div class="value" style="color:{color};">{value}</div>
            <div class="sub">{sub}</div>
        </div>""", unsafe_allow_html=True)

    churn_risk = float(row.get("predicted_churn_risk", 0))
    risk_color = "#d62728" if churn_risk > 0.15 else "#ff7f0e" if churn_risk > 0.10 else "#4caf50"

    kpi(k1, "Monthly Charges",    f"${row['monthly_charges']:.2f}",        "Current plan rate")
    kpi(k2, "Tenure",             f"{int(row['tenure_months'])} months",   f"~{row['tenure_months']/12:.1f} years")
    kpi(k3, "Churn Risk Score",   f"{churn_risk*100:.1f}%",                "OOF predicted risk", risk_color)
    kpi(k4, "Value Score",        f"{row['value_score']:.1f} / 100",       "Composite value index")
    kpi(k5, "Exp. Revenue at Risk", f"${row['expected_revenue_at_risk']:.2f}", "Annual expected loss")

    # ── Two-column layout ──
    left, right = st.columns([1, 1], gap="large")

    with left:
        # Profile
        st.markdown('<div class="section-header">👤 Customer Profile</div>', unsafe_allow_html=True)

        gender = str(row.get("gender", "N/A"))
        age_raw = row.get("age", None)
        age_str = f"{int(age_raw)} yrs" if pd.notna(age_raw) else "Unknown"
        cluster_map = {0: "Segment A — High Usage / Low Complaints", 1: "Segment B — Moderate Usage"}
        cluster_label = cluster_map.get(int(row.get("cluster", 0)), "Unknown Segment")

        profile_data = {
            "Field": ["Gender", "Age", "Region", "Contract", "Cluster / Segment",
                       "Active Months", "Complaints (Total)", "Recent Complaints (3M)"],
            "Value": [
                gender, age_str,
                str(row.get("region", "N/A")),
                str(row.get("contract_type", "N/A")),
                cluster_label,
                str(int(row.get("active_months", 0))),
                str(int(row.get("complaints_total", 0))),
                str(int(row.get("complaints_recent3", 0))),
            ]
        }
        st.dataframe(pd.DataFrame(profile_data), hide_index=True, use_container_width=True)

        # Financial
        st.markdown('<div class="section-header">💰 Financial Summary</div>', unsafe_allow_html=True)
        fin_data = {
            "Metric": ["Monthly Charges", "Annualized Revenue", "Est. Historical Revenue",
                        "Expected Annual Loss at Risk"],
            "Value": [
                f"${row['monthly_charges']:.2f}",
                f"${row.get('annualized_revenue', row['monthly_charges']*12):.2f}",
                f"${row.get('estimated_historical_revenue', 0):.2f}",
                f"${row.get('expected_revenue_at_risk', 0):.2f}",
            ]
        }
        st.dataframe(pd.DataFrame(fin_data), hide_index=True, use_container_width=True)

    with right:
        # Churn Risk Gauge
        st.markdown('<div class="section-header">📈 Churn Risk Assessment</div>', unsafe_allow_html=True)

        fig_gauge = go.Figure(go.Indicator(
            mode="gauge+number",
            value=churn_risk * 100,
            number={"suffix": "%", "font": {"size": 32, "color": "#e0e0e0"}},
            gauge={
                "axis": {"range": [0, 100], "tickfont": {"color": "#8b949e"}},
                "bar": {"color": risk_color, "thickness": 0.28},
                "bgcolor": "#1e2030",
                "borderwidth": 0,
                "steps": [
                    {"range": [0, 8],   "color": "#1e3a1e"},
                    {"range": [8, 15],  "color": "#3a2e10"},
                    {"range": [15, 100],"color": "#3a1010"},
                ],
                "threshold": {
                    "line": {"color": "white", "width": 2},
                    "thickness": 0.75,
                    "value": 7.86,
                }
            },
            title={"text": "OOF Predicted Churn Probability<br><sup>Baseline: 7.86% | White line = population avg</sup>",
                   "font": {"color": "#8b949e", "size": 12}},
        ))
        fig_gauge.update_layout(
            height=260,
            paper_bgcolor="#0f1117", plot_bgcolor="#0f1117",
            margin=dict(t=60, b=10, l=30, r=30)
        )
        st.plotly_chart(fig_gauge, use_container_width=True)

        # Usage Metrics
        st.markdown('<div class="section-header">📡 Usage Behavior</div>', unsafe_allow_html=True)
        usage_metrics = pd.DataFrame({
            "Metric": ["Avg Call Minutes/Month", "Avg Data Usage (GB)", "Avg SMS Count",
                        "Call Minutes Trend", "Data Usage Trend"],
            "Value": [
                f"{row.get('call_minutes_mean', 0):.0f} min",
                f"{row.get('data_usage_gb_mean', 0):.2f} GB",
                f"{row.get('sms_count_mean', 0):.0f}",
                f"{'📈' if row.get('call_minutes_slope', 0) > 0 else '📉'} {row.get('call_minutes_slope', 0):.2f}",
                f"{'📈' if row.get('data_usage_gb_slope', 0) > 0 else '📉'} {row.get('data_usage_gb_slope', 0):.3f}",
            ]
        })
        st.dataframe(usage_metrics, hide_index=True, use_container_width=True)

    # ── Retention Recommendation ──
    st.markdown('<div class="section-header">🎯 Retention Strategy Recommendation</div>', unsafe_allow_html=True)
    rec = str(row.get("retention_recommendation", "No recommendation available."))
    st.markdown(f'<div class="rec-box">💡 {rec}</div>', unsafe_allow_html=True)

    # ── Usage Mini-Charts ──
    st.markdown('<div class="section-header">📊 Detailed Usage Distribution Context</div>', unsafe_allow_html=True)
    c1, c2, c3 = st.columns(3)

    for col_ui, metric, label, unit in [
        (c1, "call_minutes_mean", "Avg Monthly Call Minutes", "min"),
        (c2, "data_usage_gb_mean", "Avg Monthly Data (GB)", "GB"),
        (c3, "sms_count_mean", "Avg Monthly SMS", "msgs"),
    ]:
        fig = go.Figure()
        fig.add_trace(go.Histogram(
            x=df[metric].dropna(), nbinsx=40,
            marker_color="#2a3f5f", opacity=0.8, name="All Customers"
        ))
        fig.add_vline(
            x=row.get(metric, 0), line_color="#ff7f0e", line_width=2.5,
            annotation_text=f"This customer<br>{row.get(metric,0):.1f} {unit}",
            annotation_position="top right",
            annotation_font_color="#ff7f0e"
        )
        fig.update_layout(
            title=label, height=200,
            paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
            font=dict(color="#8b949e", size=10),
            margin=dict(t=40, b=10, l=10, r=10),
            showlegend=False,
            xaxis=dict(gridcolor="#2a2d3e"),
            yaxis=dict(gridcolor="#2a2d3e"),
        )
        col_ui.plotly_chart(fig, use_container_width=True)


# ════════════════════════════════════════════════════════════════════════════════
# PAGE 2 — Retention Priority Explorer
# ════════════════════════════════════════════════════════════════════════════════
elif page == "🎯 Retention Priority Explorer":
    st.title("🎯 Retention Priority Explorer")
    st.markdown("Browse, filter, and export customers by their operational priority tier.")

    # ── Tier Summary Cards ──
    tier_stats = df.groupby("priority_tier").agg(
        count=("customer_id", "count"),
        avg_value_score=("value_score", "mean"),
        avg_churn_risk=("predicted_churn_risk", "mean"),
        total_revenue_at_risk=("expected_revenue_at_risk", "sum"),
        actual_churn_rate=("churn", "mean"),
    ).reindex(["Critical", "High", "Medium", "Low"])

    cols = st.columns(4)
    for i, (tier_name, color) in enumerate(TIER_COLORS.items()):
        row_t = tier_stats.loc[tier_name]
        with cols[i]:
            st.markdown(f"""
            <div class="metric-card" style="border-top: 3px solid {color};">
                <div class="label">{TIER_ICONS[tier_name]} {tier_name} Priority</div>
                <div class="value" style="color:{color};">{int(row_t['count']):,}</div>
                <div class="sub">customers</div>
                <hr style="border-color:#2a2d3e;margin:8px 0;">
                <div class="sub">Avg Churn Risk: <strong>{row_t['avg_churn_risk']*100:.1f}%</strong></div>
                <div class="sub">Avg Value Score: <strong>{row_t['avg_value_score']:.1f}/100</strong></div>
                <div class="sub">Revenue at Risk: <strong>${row_t['total_revenue_at_risk']:,.0f}</strong></div>
                <div class="sub">Actual Churn Rate: <strong>{row_t['actual_churn_rate']*100:.1f}%</strong></div>
            </div>
            """, unsafe_allow_html=True)

    st.markdown("---")

    # ── Filters ──
    st.markdown("### 🔧 Filter Customers")
    f1, f2, f3, f4 = st.columns([2, 2, 2, 2])

    with f1:
        selected_tiers = st.multiselect(
            "Priority Tier",
            options=["Critical", "High", "Medium", "Low"],
            default=["Critical", "High"],
            help="Filter customers by their operational retention priority tier"
        )
    with f2:
        contract_opts = ["All"] + sorted(df["contract_type"].dropna().unique().tolist())
        selected_contract = st.selectbox("Contract Type", contract_opts)

    with f3:
        region_opts = ["All"] + sorted(df["region"].dropna().unique().tolist())
        selected_region = st.selectbox("Region", region_opts)

    with f4:
        churn_filter = st.selectbox(
            "Actual Churn Status",
            options=["All", "Churned (1)", "Retained (0)"],
        )

    # Numeric sliders
    sl1, sl2 = st.columns(2)
    with sl1:
        min_charge, max_charge = float(df["monthly_charges"].min()), float(df["monthly_charges"].max())
        charge_range = st.slider("Monthly Charges ($)", min_charge, max_charge, (min_charge, max_charge))
    with sl2:
        min_risk = float(df["predicted_churn_risk"].min())
        max_risk = float(df["predicted_churn_risk"].max())
        risk_range = st.slider("Predicted Churn Risk (%)", 0.0, 100.0,
                               (min_risk * 100, max_risk * 100))

    # Apply filters
    mask = df["priority_tier"].isin(selected_tiers if selected_tiers else ["Critical", "High", "Medium", "Low"])
    if selected_contract != "All":
        mask &= df["contract_type"] == selected_contract
    if selected_region != "All":
        mask &= df["region"] == selected_region
    if churn_filter == "Churned (1)":
        mask &= df["churn"] == 1
    elif churn_filter == "Retained (0)":
        mask &= df["churn"] == 0
    mask &= (df["monthly_charges"] >= charge_range[0]) & (df["monthly_charges"] <= charge_range[1])
    mask &= (df["predicted_churn_risk"] * 100 >= risk_range[0]) & (df["predicted_churn_risk"] * 100 <= risk_range[1])

    filtered = df[mask].copy()

    # Sort
    sort_col, sort_dir_col = st.columns([3, 1])
    with sort_col:
        sort_by = st.selectbox(
            "Sort by",
            options=["retention_rank", "predicted_churn_risk", "value_score",
                     "expected_revenue_at_risk", "monthly_charges", "tenure_months"],
            index=0,
            format_func=lambda x: x.replace("_", " ").title()
        )
    with sort_dir_col:
        sort_asc = st.selectbox("Direction", ["Ascending", "Descending"], index=0 if sort_by == "retention_rank" else 1)

    filtered = filtered.sort_values(sort_by, ascending=(sort_asc == "Ascending"))

    st.markdown(f"**{len(filtered):,} customers** match your filters (of {len(df):,} total)")

    # ── Results Table ──
    display_cols = {
        "retention_rank": "Rank",
        "customer_id": "Customer ID",
        "priority_tier": "Tier",
        "monthly_charges": "Monthly ($)",
        "tenure_months": "Tenure (Mo)",
        "contract_type": "Contract",
        "region": "Region",
        "value_score": "Value Score",
        "predicted_churn_risk": "Churn Risk",
        "expected_revenue_at_risk": "Rev. at Risk ($)",
        "churn": "Actual Churn",
        "retention_recommendation": "Recommendation",
    }
    table_df = filtered[[c for c in display_cols if c in filtered.columns]].rename(columns=display_cols)
    if "Churn Risk" in table_df.columns:
        table_df["Churn Risk"] = (table_df["Churn Risk"] * 100).round(1).astype(str) + "%"
    if "Value Score" in table_df.columns:
        table_df["Value Score"] = table_df["Value Score"].round(1)
    if "Monthly ($)" in table_df.columns:
        table_df["Monthly ($)"] = table_df["Monthly ($)"].round(2)
    if "Rev. at Risk ($)" in table_df.columns:
        table_df["Rev. at Risk ($)"] = table_df["Rev. at Risk ($)"].round(2)

    st.dataframe(table_df.head(200), hide_index=True, use_container_width=True)
    if len(filtered) > 200:
        st.caption(f"Showing first 200 of {len(filtered):,} matching customers.")

    # Download button
    csv_bytes = filtered[[c for c in display_cols if c in filtered.columns]].rename(columns=display_cols).to_csv(index=False).encode("utf-8")
    st.download_button(
        label=f"⬇️ Download filtered list ({len(filtered):,} customers)",
        data=csv_bytes,
        file_name="filtered_retention_priority.csv",
        mime="text/csv",
        use_container_width=True,
    )

    # ── Tier Distribution Charts ──
    st.markdown("---")
    st.markdown("### 📈 Filtered Cohort Analysis")
    ch1, ch2, ch3 = st.columns(3)

    with ch1:
        tier_count = filtered["priority_tier"].value_counts().reindex(["Critical", "High", "Medium", "Low"]).fillna(0)
        fig_pie = px.pie(
            values=tier_count.values,
            names=tier_count.index,
            color=tier_count.index,
            color_discrete_map=TIER_COLORS,
            title="Tier Distribution (filtered)",
            hole=0.45,
        )
        fig_pie.update_layout(paper_bgcolor="#0f1117", plot_bgcolor="#0f1117",
                              font=dict(color="#c9d1d9"), height=280, margin=dict(t=40,b=10))
        st.plotly_chart(fig_pie, use_container_width=True)

    with ch2:
        fig_box = px.box(
            filtered, x="priority_tier", y="predicted_churn_risk",
            category_orders={"priority_tier": ["Critical", "High", "Medium", "Low"]},
            color="priority_tier", color_discrete_map=TIER_COLORS,
            title="Churn Risk Distribution by Tier",
        )
        fig_box.update_layout(paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
                              font=dict(color="#c9d1d9"), height=280,
                              showlegend=False, margin=dict(t=40,b=10),
                              xaxis=dict(gridcolor="#2a2d3e"),
                              yaxis=dict(gridcolor="#2a2d3e", tickformat=".0%"))
        st.plotly_chart(fig_box, use_container_width=True)

    with ch3:
        fig_scatter = px.scatter(
            filtered.sample(min(1000, len(filtered)), random_state=42),
            x="value_score", y="predicted_churn_risk",
            color="priority_tier", color_discrete_map=TIER_COLORS,
            opacity=0.6, size_max=6,
            title="Value Score vs. Churn Risk",
            labels={"value_score": "Value Score", "predicted_churn_risk": "Churn Risk"},
        )
        fig_scatter.update_layout(paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
                                  font=dict(color="#c9d1d9"), height=280,
                                  margin=dict(t=40,b=10),
                                  xaxis=dict(gridcolor="#2a2d3e"),
                                  yaxis=dict(gridcolor="#2a2d3e", tickformat=".0%"))
        st.plotly_chart(fig_scatter, use_container_width=True)


# ════════════════════════════════════════════════════════════════════════════════
# PAGE 3 — Model & Portfolio Summary
# ════════════════════════════════════════════════════════════════════════════════
elif page == "📊 Model & Portfolio Summary":
    st.title("📊 Model & Portfolio Summary")
    st.markdown("Churn model performance overview, feature importance, and portfolio-level retention economics.")

    fi_rf, fi_lr, model_cmp = load_model_data()

    # ── Portfolio KPIs ──
    st.markdown("### 💼 Portfolio Financial Overview")
    pk1, pk2, pk3, pk4, pk5 = st.columns(5)

    total_annual   = df["annualized_revenue"].sum()
    total_exp_loss = df["expected_revenue_at_risk"].sum()
    total_hist     = df["estimated_historical_revenue"].sum()
    churn_base     = df["churn"].mean()
    imputed_n      = 54  # from notebook 10 analysis

    def pkpi(col, label, value, sub="", color="#58a6ff"):
        col.markdown(f"""
        <div class="metric-card">
            <div class="label">{label}</div>
            <div class="value" style="color:{color};">{value}</div>
            <div class="sub">{sub}</div>
        </div>""", unsafe_allow_html=True)

    pkpi(pk1, "Total Customers",        f"{len(df):,}",                  f"{imputed_n} imputed charges")
    pkpi(pk2, "Annualized Billing",     f"${total_annual/1e6:.2f}M",     "Portfolio run-rate")
    pkpi(pk3, "Expected Loss at Risk",  f"${total_exp_loss/1e3:.1f}K",   f"{total_exp_loss/total_annual*100:.2f}% of billing", "#d62728")
    pkpi(pk4, "Est. Historical Revenue",f"${total_hist/1e6:.1f}M",       "Cumulative past billing")
    pkpi(pk5, "Baseline Churn Rate",    f"{churn_base*100:.2f}%",        "786 churned / 10,000", "#ff7f0e")

    st.markdown("---")

    # ── Model Performance ──
    st.markdown("### 🤖 Churn Model Performance (Test Set)")

    tuned = model_cmp[model_cmp["Decision Rule"] == "Train-OOF Tuned"].copy()
    tuned = tuned[["Model", "ROC-AUC", "PR-AUC (AP)", "Precision", "Recall", "F1", "Brier", "Threshold"]].round(4)
    st.dataframe(tuned.reset_index(drop=True), hide_index=True, use_container_width=True)

    st.markdown("""
    <div class="rec-box">
    ⚠️ <strong>Model Diagnostic:</strong> All three models achieve ROC-AUC ≈ 0.49–0.49 and PR-AUC ≈ 0.077–0.081
    against a baseline prevalence of 7.86%. This confirms near-random discrimination capability. The pipeline is
    presented as an <em>experimental prototype</em>. Customer Value Score prioritization remains the primary
    actionable signal; predicted churn risk adds marginal information.
    </div>
    """, unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### 🔍 Feature Importance — Random Forest")

    m1, m2 = st.columns([2, 1])

    with m1:
        top_n = st.slider("Show top N features", 5, 36, 15)
        fi_plot = fi_rf.head(top_n).sort_values("importance")
        fig_fi = px.bar(
            fi_plot, x="importance", y="feature_clean",
            orientation="h", color="importance",
            color_continuous_scale="Blues",
            title=f"Top {top_n} Features by Random Forest Importance",
            labels={"importance": "Importance Score", "feature_clean": "Feature"},
        )
        fig_fi.update_layout(
            paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
            font=dict(color="#c9d1d9"),
            coloraxis_showscale=False,
            height=max(350, top_n * 25),
            margin=dict(t=50, b=10, l=10, r=10),
            xaxis=dict(gridcolor="#2a2d3e"),
            yaxis=dict(gridcolor="#2a2d3e"),
        )
        st.plotly_chart(fig_fi, use_container_width=True)

    with m2:
        st.markdown("**Top 10 Features (RF)**")
        st.dataframe(
            fi_rf[["feature_clean", "importance"]].head(10).rename(
                columns={"feature_clean": "Feature", "importance": "Importance"}
            ).round(4),
            hide_index=True, use_container_width=True
        )
        st.markdown("---")
        st.markdown("**Top 10 Features (LR Coefficients)**")
        st.dataframe(
            fi_lr[["feature_clean", "coefficient"]].head(10).rename(
                columns={"feature_clean": "Feature", "coefficient": "Coefficient"}
            ).round(4),
            hide_index=True, use_container_width=True
        )

    # ── Priority Tier Revenue Breakdown ──
    st.markdown("---")
    st.markdown("### 🎯 Retention Campaign Economics by Tier")

    outreach_costs = {"Critical": 25.0, "High": 12.0, "Medium": 3.0, "Low": 0.5}
    econ = df.groupby("priority_tier").agg(
        customers=("customer_id", "count"),
        total_billing=("annualized_revenue", "sum"),
        exp_revenue_at_risk=("expected_revenue_at_risk", "sum"),
        avg_value_score=("value_score", "mean"),
        actual_churn=("churn", "sum"),
    ).reindex(["Critical", "High", "Medium", "Low"])

    econ["outreach_cost_per_cust"] = econ.index.map(outreach_costs)
    econ["total_outreach_cost"] = econ["customers"] * econ["outreach_cost_per_cust"]
    econ["expected_saved_margin"] = econ["exp_revenue_at_risk"] * 0.60 * 0.15
    econ["net_benefit"] = econ["expected_saved_margin"] - econ["total_outreach_cost"]

    econ_display = econ[["customers", "total_billing", "exp_revenue_at_risk",
                          "total_outreach_cost", "expected_saved_margin", "net_benefit"]].copy()
    econ_display.columns = ["Customers", "Annual Billing ($)", "Exp. Loss at Risk ($)",
                             "Campaign Cost ($)", "Saved Margin ($)", "Net Benefit ($)"]
    econ_display = econ_display.round(2)
    st.dataframe(econ_display, use_container_width=True)

    # Net benefit bar chart
    fig_nb = px.bar(
        econ.reset_index(), x="priority_tier", y="net_benefit",
        color="priority_tier", color_discrete_map=TIER_COLORS,
        title="Expected Net Campaign Benefit by Priority Tier ($)",
        labels={"priority_tier": "Tier", "net_benefit": "Net Benefit ($)"},
        category_orders={"priority_tier": ["Critical", "High", "Medium", "Low"]},
    )
    fig_nb.update_layout(
        paper_bgcolor="#0f1117", plot_bgcolor="#1e2030",
        font=dict(color="#c9d1d9"), height=300, showlegend=False,
        margin=dict(t=50,b=10), xaxis=dict(gridcolor="#2a2d3e"),
        yaxis=dict(gridcolor="#2a2d3e")
    )
    st.plotly_chart(fig_nb, use_container_width=True)

    st.caption("Assumptions: 60% gross margin rate, 15% retention offer success rate. These are illustrative parameters and should be calibrated with actual A/B test results.")

Overwriting ../app/app.py


---
## ▶ Launch the Live App

Run the cell below to start the Streamlit server, then open  
**http://localhost:8501** in your browser.

---
## Milestone 15 — Summary

| Deliverable | Status |
|---|---|
| `app/app.py` — production Streamlit application | ✅ Built (719 lines) |
| Page 1: Customer Lookup with churn gauge + usage charts | ✅ |
| Page 2: Retention Priority Explorer with tier filters + CSV export | ✅ |
| Page 3: Model & Portfolio Summary with feature importance + economics | ✅ |
| Notebook inline demo of all pages | ✅ |

**Next:** Milestone 16 — Final Documentation, Testing & CV Portfolio Packaging

In [18]:
import subprocess, sys, os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
app_path     = project_root / "app" / "app.py"
streamlit_exe = project_root / ".venv" / "Scripts" / "streamlit.exe"

print(f"App    : {app_path}")
print(f"Binary : {streamlit_exe}")
print()

if not streamlit_exe.exists():
    print("ERROR: .venv/Scripts/streamlit.exe not found.")
    print("Activate your virtualenv and install: pip install streamlit")
else:
    print("Starting Streamlit...")
    print("Open http://localhost:8501 in your browser.")
    print("Press CTRL+C (or interrupt the kernel) to stop.")
    subprocess.run(
        [str(streamlit_exe), "run", str(app_path),
         "--server.port", "8501",
         "--server.headless", "true"],
        cwd=str(project_root),
    )

App    : d:\Ahmed\My Projects\telecom intelligence project\app\app.py
Binary : d:\Ahmed\My Projects\telecom intelligence project\.venv\Scripts\streamlit.exe

Starting Streamlit...
Open http://localhost:8501 in your browser.
Press CTRL+C (or interrupt the kernel) to stop.
